# Lab 4 — YouTube Video Summarizer with NGROK + FastAPI
This notebook rebuilds Lab 1 as a Kaggle backend. Streamlit runs locally and sends requests through NGROK.


In [1]:
# Lab 4 — Kaggle Backend: YouTube Summarizer + FastAPI + NGROK

# 1) Install dependencies
!pip install -q youtube-transcript-api transformers sentencepiece arabert langdetect fastapi uvicorn pyngrok accelerate

import re, socket, threading, time, torch
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import BertTokenizer, AutoTokenizer, AutoModelForSeq2SeqLM
from arabert.preprocess import ArabertPreprocessor
from langdetect import detect

# ====== PUT YOUR DATA HERE ======
NGROK_TOKEN = "PUT_YOUR_NEW_NGROK_TOKEN_HERE"
API_KEY = "PUT_YOUR_API_KEY_HERE!"

# ====== Lab 1 pipeline ======
def extract_video_id(url):
    parsed=urlparse(url); host=(parsed.hostname or '').lower()
    if host.startswith('www.'): host=host[4:]
    if host=='youtu.be':
        vid=parsed.path.lstrip('/').split('/')[0]
        if vid: return vid
    if host in ('youtube.com','m.youtube.com','music.youtube.com'):
        qs=parse_qs(parsed.query)
        if qs.get('v'): return qs['v'][0]
        parts=parsed.path.strip('/').split('/')
        if len(parts)>=2 and parts[0] in ('shorts','embed','live','v'): return parts[1]
    raise ValueError('Invalid YouTube URL.')

def fetch_transcript(video_id):
    api=YouTubeTranscriptApi()
    fetched=api.fetch(video_id, languages=['ar','en'])
    return '\n'.join(x.text for x in fetched)

def clean_transcript(text):
    return re.sub(r'\s+',' ',text).strip()

def detect_language(text):
    try:
        lang=detect(text)
        return lang if lang in ('ar','en') else lang
    except Exception:
        return 'unknown'

def split_text(text,max_words=350):
    words=text.split()
    return [' '.join(words[i:i+max_words]) for i in range(0,len(words),max_words)]


# ====== Arabic Model ======
ARABIC_MODEL_NAME='malmarjeh/mbert2mbert-arabic-text-summarization'

arabic_preprocessor=ArabertPreprocessor(model_name='')
arabic_tokenizer=BertTokenizer.from_pretrained(ARABIC_MODEL_NAME)
arabic_model=AutoModelForSeq2SeqLM.from_pretrained(ARABIC_MODEL_NAME)

# Use GPU if available, otherwise CPU
arabic_device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

arabic_model=arabic_model.to(arabic_device)
arabic_model.eval()


def summarize_arabic(text,max_length=150):

    processed_text=arabic_preprocessor.preprocess(text)

    inputs=arabic_tokenizer(
        processed_text,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )

    inputs={
        k:v.to(arabic_device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        summary_ids=arabic_model.generate(
            **inputs,
            max_length=max_length,
            min_length=30,
            num_beams=3,
            repetition_penalty=3.0,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            do_sample=False,
            early_stopping=True,
            pad_token_id=arabic_tokenizer.eos_token_id
        )

    return arabic_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )


# ====== Test Arabic Model ======
arabic_test_text='''
تنظيم الوقت يساعد الطلاب على إنجاز المهام اليومية
بشكل أفضل. يمكن تقسيم وقت الدراسة إلى فترات قصيرة
مع أخذ فترات راحة بسيطة بينها. كما يساعد وضع قائمة
بالأهداف اليومية على التركيز وإنهاء المهام في موعدها.
'''

print('Arabic model test:')

print(
    summarize_arabic(
        arabic_test_text
    )
)


# ====== English Model ======
ENGLISH_MODEL_NAME='sshleifer/distilbart-cnn-12-6'

english_tokenizer=AutoTokenizer.from_pretrained(ENGLISH_MODEL_NAME)
english_model=AutoModelForSeq2SeqLM.from_pretrained(ENGLISH_MODEL_NAME)

device=torch.device('cpu')
english_model=english_model.to(device)
english_model.eval()


def summarize_english(text,max_input_tokens=900):

    inputs=english_tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_input_tokens
    )

    inputs={
        k:v.to(device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        ids=english_model.generate(
            **inputs,
            max_length=150,
            min_length=30,
            num_beams=4,
            no_repeat_ngram_size=3,
            do_sample=False,
            early_stopping=True
        )

    return english_tokenizer.decode(
        ids[0],
        skip_special_tokens=True
    )


# ====== Summarize Chunks ======
def summarize_transcript(chunks,language):

    out=[]

    for i,chunk in enumerate(chunks):

        print(
            f'Processing chunk {i+1}/{len(chunks)}...'
        )

        if language=='ar':

            summary=summarize_arabic(
                chunk
            )

        elif language=='en':

            summary=summarize_english(
                chunk
            )

        else:

            raise ValueError(
                'Only Arabic and English are supported.'
            )

        out.append(
            summary
        )

        print(
            f'Chunk {i+1}/{len(chunks)} completed.'
        )

    print(
        'All chunks processed successfully.'
    )

    return out


# ====== Final Summary ======
def generate_final_summary(text,language):

    if language=='ar':
        return summarize_arabic(text)

    if language=='en':
        return summarize_english(text)

    raise ValueError(
        'Only Arabic and English are supported.'
    )


# ====== Complete YouTube Summarization Pipeline ======
def summarize_youtube_video(youtube_url,max_words=350):

    print(
        'Starting YouTube summarization...'
    )

    video_id=extract_video_id(
        youtube_url
    )

    print(
        'Video ID:',
        video_id
    )

    print(
        'Fetching transcript...'
    )

    transcript=fetch_transcript(
        video_id
    )

    print(
        'Transcript fetched successfully.'
    )

    cleaned=clean_transcript(
        transcript
    )

    language=detect_language(
        cleaned
    )

    print(
        'Detected language:',
        language
    )

    if language not in ('ar','en'):

        raise ValueError(
            f'Unsupported transcript language: {language}'
        )

    chunks=split_text(
        cleaned,
        max_words
    )

    print(
        f'Total chunks: {len(chunks)}'
    )

    chunk_summaries=summarize_transcript(
        chunks,
        language
    )

    # ====== Combined Summary ======
    print(
        'Generating combined summary...'
    )

    combined_summary=' '.join(
        chunk_summaries
    )

    print(
        'Combined summary generated successfully.'
    )

    # ====== Final Summary ======
    print(
        'Generating final summary...'
    )

    final_summary=generate_final_summary(
        combined_summary,
        language
    )

    print(
        'Final summary generated successfully.'
    )

    return {
        'video_id':video_id,
        'language':language,
        'transcript_characters':len(transcript),
        'chunk_count':len(chunks),
        'combined_summary':combined_summary,
        'summary':final_summary
    }


# ====== FastAPI + NGROK ======
from fastapi import FastAPI, Request, HTTPException
import uvicorn
from pyngrok import ngrok, conf


if NGROK_TOKEN.startswith('PUT_') or API_KEY.startswith('PUT_'):

    raise ValueError(
        'Set NGROK_TOKEN and API_KEY first.'
    )


app=FastAPI(
    title='Lab 4 YouTube Summarizer API',
    version='1.0'
)


@app.get('/')
async def root():

    return {
        'status':'running',
        'service':'Lab 4 YouTube Video Summarizer'
    }


@app.post('/summarize')
async def summarize_endpoint(req:Request):

    if req.headers.get('authorization') != f'Bearer {API_KEY}':

        print(
            '401 Unauthorized'
        )

        raise HTTPException(
            401,
            'Unauthorized'
        )

    data=await req.json()

    youtube_url=data.get(
        'youtube_url',
        ''
    )

    max_words=int(
        data.get(
            'max_words',
            350
        )
    )

    if not youtube_url:

        raise HTTPException(
            400,
            'youtube_url is required'
        )

    try:

        result=summarize_youtube_video(
            youtube_url,
            max_words
        )

        print(
            '200 OK - Request completed successfully.'
        )

        return result

    except Exception as e:

        print(
            '500 Internal Server Error:',
            str(e)
        )

        raise HTTPException(
            500,
            str(e)
        )


def free_port():

    s=socket.socket()

    s.bind(
        ('',0)
    )

    port=s.getsockname()[1]

    s.close()

    return port


port=free_port()

conf.get_default().auth_token=NGROK_TOKEN

public_url=ngrok.connect(
    port
).public_url


print(
    'NGROK PUBLIC URL:',
    public_url
)

print(
    'API ENDPOINT:',
    public_url+'/summarize'
)


def run_api():

    uvicorn.run(
        app,
        host='0.0.0.0',
        port=port
    )


threading.Thread(
    target=run_api,
    daemon=True
).start()

time.sleep(2)

print(
    'FastAPI server started on port:',
    port
)

# Keep the Kaggle session running.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.1 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.9 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)


tokenizer_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:49903 (Press CTRL+C to quit)


NGROK PUBLIC URL: https://durable-whiny-issue.ngrok-free.dev
API ENDPOINT: https://durable-whiny-issue.ngrok-free.dev/summarize
FastAPI server started on port: 49903
